# Tutorial: Adding a New Model to GFMBench-API

This notebook shows how to plug **your own model** into GFMBench-API for evaluation.

**What you will do**

1. Review the duck-typed model API (which methods tasks call).
2. Implement a **random mock** model and evaluate it on:
   - **BEND expression** (zero-shot variant effect)
   - **GUE promoter** (supervised single-sequence classification), with a tiny in-notebook linear probe
3. Implement a simplified **Nucleotide Transformer v3 (8M)** adapter the same way, and re-run both tasks.

Tasks come from `gfmbench_api`. Models, probing, and helpers are implemented **in this notebook** so the pattern is easy to copy.

> Inheritance from `BaseGFMModel` is **optional**. GFMBench-API uses duck typing: implement the methods your tasks need; return `None` for the rest (dependent metrics are skipped).


## 0. Setup

Install the package requirements from the repo root, then add the Hugging Face libraries used by the real-model section:

```bash
pip install -r basic_requirements.txt
pip install transformers
```

The real model in Part B (`InstaDeepAI/NTv3_8M_pre`) is **gated**: accept its license on Hugging Face and run `huggingface-cli login` first.

Then set `ROOT_DATA_DIR` in the next cell before running anything else.


### User config — update this path

Set `ROOT_DATA_DIR` to a writable directory where GFMBench-API can store task data (and download reference genomes / datasets on first run).

In [ ]:
# >>> UPDATE THIS PATH <<<
ROOT_DATA_DIR = "/path/to/data"

print(f"ROOT_DATA_DIR = {ROOT_DATA_DIR}")


In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Third-party sources used later in this notebook (see also THIRD_PARTY_NOTICES.md):
# - https://huggingface.co/InstaDeepAI/NTv3_8M_pre — gated; accept license on HuggingFace
# - https://huggingface.co/datasets/leannmlindsey/GUE — MIT
# - https://sid.erda.dk/share_redirect/aNQa0Oz2lY/data/variant_effects/variant_effects_expression.bed — BSD-3-Clause

from __future__ import annotations
import sys

from typing import List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_SAMPLES = None  # use full datasets (set an int to cap for a quicker dry run)
PROBE_EPOCHS = 3
BATCH_SIZE = 32

TASK_CONFIG = {
    "max_sequence_length": 512,
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "max_num_samples": MAX_SAMPLES,
}

print(f"device={DEVICE}, max_samples={MAX_SAMPLES or 'all'}")


## 1. What GFMBench-API expects from a model

See `gfmbench_api/tasks/base/base_gfm_model.py` for full docstrings and tensor shapes.

| Method | Used by (in this tutorial) | Notes |
|:-------|:---------------------------|:------|
| `infer_sequence_to_sequence` | BEND (embeddings / cosine); also embeddings for linear probe | Returns `(seq_probs, seq_embeddings, seq_representative)` — any can be `None` |
| `sequence_pos_to_prob_pos` | BEND SNV position metrics | Maps a DNA base index → model output index |
| `infer_masked_sequence_to_token_probs` | BEND masked-LLR metrics | Optional; return `(None, None)` to skip those metrics |
| `infer_sequence_to_labels_probs` | GUE after you attach a classification head | Shape `[batch, num_classes]` for binary promoter |
| `infer_variant_ref_sequences_to_labels_probs` | *(not used here)* | Return `None` |
| `infer_sequence_to_regression` | *(not used here)* | Return `None` |

**Rule of thumb:** implement what your tasks need; stub the rest with `None`.


## 2. Part A — Random mock model

A throwaway model that returns random arrays with the right shapes. Useful to wire up tasks before your real forward pass exists.


In [ ]:
class RandomMockModel:
    """Minimal duck-typed model: random outputs, correct shapes."""

    def __init__(self, device: str = "cpu", hidden_dim: int = 32, num_classes: int = 2):
        self.device = device
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self._clf_head: Optional[nn.Linear] = None  # set by linear_probe_gue

    def get_hidden_dim(self) -> int:
        return self.hidden_dim

    def eval(self):
        if self._clf_head is not None:
            self._clf_head.eval()
        return self

    def train(self, mode: bool = True):
        if self._clf_head is not None:
            self._clf_head.train(mode)
        return self

    # --- used by BEND (zero-shot VEP) ---

    def infer_sequence_to_sequence(
        self, sequences: List[str], conditional_input=None
    ) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
        n = len(sequences)
        max_len = max((len(s) for s in sequences), default=1)
        # seq_probs: per-base scores [N, L]
        seq_probs = np.random.rand(n, max_len).astype(np.float32)
        # per-position embeddings [N, L, H]
        embeds = np.random.randn(n, max_len, self.hidden_dim).astype(np.float32)
        # sequence representative [N, H]
        reps = np.random.randn(n, self.hidden_dim).astype(np.float32)
        return seq_probs, embeds, reps

    def sequence_pos_to_prob_pos(self, sequences: List[str], pos: int) -> np.ndarray:
        # Identity mapping: DNA index == output index (when in range).
        return np.array(
            [pos if 0 <= pos < len(s) else -1 for s in sequences], dtype=np.int64
        )

    def infer_masked_sequence_to_token_probs(
        self,
        sequences: List[str],
        variant_pos: int,
        variant_letters: List[str],
        reference_letters: List[str],
        conditional_input=None,
    ) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        n = len(sequences)
        return (
            np.random.rand(n).astype(np.float32),
            np.random.rand(n).astype(np.float32),
        )

    # --- used by GUE (after optional linear probe) ---

    def infer_sequence_to_labels_probs(
        self, sequences: List[str], conditional_input=None
    ) -> Optional[np.ndarray]:
        n = len(sequences)
        if self._clf_head is None:
            # Random class probs so eval still runs before probing.
            logits = np.random.randn(n, self.num_classes).astype(np.float32)
            exp = np.exp(logits - logits.max(axis=1, keepdims=True))
            return exp / exp.sum(axis=1, keepdims=True)

        _, _, reps = self.infer_sequence_to_sequence(sequences)
        with torch.no_grad():
            x = torch.from_numpy(reps).to(self.device)
            probs = F.softmax(self._clf_head(x), dim=-1)
        return probs.cpu().numpy()

    # --- not needed for BEND expression / GUE promoter ---

    def infer_variant_ref_sequences_to_labels_probs(
        self, variant_sequences, ref_sequences, conditional_input=None
    ):
        return None  # supervised variant-pair tasks only

    def infer_sequence_to_regression(self, sequences, conditional_input=None):
        return None  # regression tasks only


mock = RandomMockModel(device=DEVICE)
print("RandomMockModel ready, hidden_dim =", mock.get_hidden_dim())


### Shared helper: tiny linear probe for GUE

Supervised single-seq tasks call `infer_sequence_to_labels_probs`. Below we train a **host-side** `nn.Linear` on frozen sequence representatives — intentionally minimal (not the production `GFMFinetuner`).


In [ ]:
def _embed_reps(model, sequences: List[str]) -> torch.Tensor:
    """Get sequence representatives as a float32 tensor on model.device."""
    _, _, reps = model.infer_sequence_to_sequence(sequences)
    if reps is None:
        raise RuntimeError("Model must return sequence_representative for linear probing")
    return torch.from_numpy(np.asarray(reps, dtype=np.float32)).to(model.device)


def linear_probe_gue(model, task, epochs: int = PROBE_EPOCHS, lr: float = 1e-2):
    """Train a 2-class linear head on GUE train embeddings; attach it to the model.

    Expects the model to expose:
      - get_hidden_dim()
      - infer_sequence_to_sequence(...) -> (..., sequence_representative)
      - attribute `_clf_head` used by infer_sequence_to_labels_probs
    """
    train_ds = task.get_finetune_dataset()
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    head = nn.Linear(model.get_hidden_dim(), 2).to(model.device)
    opt = torch.optim.Adam(head.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    head.train()
    for epoch in range(epochs):
        total, n = 0.0, 0
        for sequences, labels, _cond in loader:
            labels = labels.to(model.device).long()
            with torch.no_grad():
                reps = _embed_reps(model, list(sequences))
            logits = head(reps)
            loss = loss_fn(logits, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * labels.size(0)
            n += labels.size(0)
        print(f"  probe epoch {epoch + 1}/{epochs}  loss={total / max(n, 1):.4f}")

    head.eval()
    model._clf_head = head
    return model


### A.1 Evaluate mock on BEND expression (zero-shot)

Uses `infer_sequence_to_sequence`, `sequence_pos_to_prob_pos`, and (optionally) `infer_masked_sequence_to_token_probs`. Expect near-chance scores from a random model.


In [ ]:
from gfmbench_api.tasks.concrete.bend_vep_expression_task import BendVEPExpression

bend_task = BendVEPExpression(root_data_dir_path=ROOT_DATA_DIR, task_config=TASK_CONFIG)
print("Task:", bend_task.get_task_name())
print("Attributes:", bend_task.get_task_attributes())

mock.eval()
bend_scores_mock = bend_task.eval_test_set(mock)
print("BEND scores (RandomMock):")
for k, v in bend_scores_mock.items():
    print(f"  {k}: {v}")


### A.2 Probe + evaluate mock on GUE promoter


In [ ]:
from gfmbench_api.tasks.concrete.gue_promoter_all_task import GuePromoterAllTask

gue_task = GuePromoterAllTask(root_data_dir_path=ROOT_DATA_DIR, task_config=TASK_CONFIG)
print("Task:", gue_task.get_task_name())
print("Attributes:", gue_task.get_task_attributes())

print("Linear probing RandomMock on GUE train split...")
linear_probe_gue(mock, gue_task)

mock.eval()
gue_scores_mock = gue_task.eval_test_set(mock)
print("GUE scores (RandomMock + probe):")
for k, v in gue_scores_mock.items():
    print(f"  {k}: {v}")


## 3. Part B — Real model (Nucleotide Transformer v3, 8M)

We use [`InstaDeepAI/NTv3_8M_pre`](https://huggingface.co/InstaDeepAI/NTv3_8M_pre) — small, fast, and single-nucleotide tokenized, which keeps position mapping trivial.

The checkpoint is **gated**: accept the license on Hugging Face and log in (`huggingface-cli login`) before running this section.

| Method | Status |
|:-------|:-------|
| `infer_sequence_to_sequence` | **Implemented** (last hidden states + mean-pooled representative) |
| `sequence_pos_to_prob_pos` | **Implemented** (identity: one token per nucleotide) |
| `infer_masked_sequence_to_token_probs` | **Implemented** (mask the SNV token, read A/T/C/G probabilities) |
| `infer_sequence_to_labels_probs` | Starts as `None`; filled once `linear_probe_gue` attaches `_clf_head` |
| `infer_variant_ref_sequences_to_labels_probs` | **Not implemented** (`None`) — supervised variant-pair tasks |
| `infer_sequence_to_regression` | **Not implemented** (`None`) — regression tasks |

The backbone stays frozen; only the probe head trains.

Two NTv3 details worth copying for your own model:

- Sequence length must be padded to a multiple of **128** tokens.
- Tokenize with `add_special_tokens=False`, so token index == nucleotide index.


In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer

NTV3_SEQ_MULTIPLE = 128  # NTv3 requires the token length to be a multiple of 128


class SimpleNTv3Model:
    """Tutorial adapter: NTv3 8M with the GFMBench-API methods this demo needs."""

    HF_NAME = "InstaDeepAI/NTv3_8M_pre"  # gated: accept the license on HuggingFace

    def __init__(self, device: str = "cpu", max_length: int = 1024):
        self.device = device
        self.max_length = max_length
        self._clf_head: Optional[nn.Linear] = None

        print(f"Loading {self.HF_NAME} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.HF_NAME, trust_remote_code=True)
        self.model = AutoModelForMaskedLM.from_pretrained(self.HF_NAME, trust_remote_code=True)
        self.model.to(device).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.hidden_dim = int(self.model.core.config.embed_dim)
        print(f"Loaded. hidden_dim={self.hidden_dim}")

    def get_hidden_dim(self) -> int:
        return self.hidden_dim

    def eval(self):
        self.model.eval()
        if self._clf_head is not None:
            self._clf_head.eval()
        return self

    def train(self, mode: bool = True):
        # Backbone stays frozen; only the probe head toggles.
        if self._clf_head is not None:
            self._clf_head.train(mode)
        return self

    def _encode(self, sequences: Sequence[str]):
        enc = self.tokenizer(
            list(sequences),
            add_special_tokens=False,  # token index == nucleotide index
            padding=True,
            pad_to_multiple_of=NTV3_SEQ_MULTIPLE,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc.get("attention_mask", torch.ones_like(input_ids)).to(self.device)
        return input_ids, attention_mask

    def _forward_hidden(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        return outputs

    def _mean_pool(self, hidden, attention_mask):
        mask = attention_mask.float().unsqueeze(-1).expand(hidden.size())
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    def infer_sequence_to_sequence(
        self, sequences: List[str], conditional_input=None
    ) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
        input_ids, attention_mask = self._encode(sequences)
        with torch.no_grad():
            hidden = self._forward_hidden(input_ids, attention_mask)["hidden_states"][-1]
            reps = self._mean_pool(hidden, attention_mask)
        seq_probs = None  # this tutorial skips per-base LM probabilities
        return seq_probs, hidden.float().cpu().numpy(), reps.float().cpu().numpy()

    def sequence_pos_to_prob_pos(self, sequences: List[str], pos: int) -> np.ndarray:
        # One token per nucleotide and no special tokens -> identity mapping.
        return np.full(len(sequences), pos, dtype=np.int64)

    def infer_masked_sequence_to_token_probs(
        self,
        sequences: List[str],
        variant_pos: int,
        variant_letters: List[str],
        reference_letters: List[str],
        conditional_input=None,
    ) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        input_ids, attention_mask = self._encode(sequences)
        if variant_pos >= input_ids.shape[1]:
            return None, None

        masked = input_ids.clone()
        masked[:, variant_pos] = self.tokenizer.mask_token_id

        with torch.no_grad():
            logits = self._forward_hidden(masked, attention_mask)["logits"].float()
            probs = torch.softmax(logits, dim=-1)

        var_p, ref_p = [], []
        unk = self.tokenizer.unk_token_id
        for i in range(len(sequences)):
            vid = self.tokenizer.convert_tokens_to_ids(variant_letters[i].upper())
            rid = self.tokenizer.convert_tokens_to_ids(reference_letters[i].upper())
            if vid == unk or rid == unk:
                var_p.append(0.0)
                ref_p.append(0.0)
                continue
            var_p.append(probs[i, variant_pos, vid].item())
            ref_p.append(probs[i, variant_pos, rid].item())
        return np.asarray(var_p, dtype=np.float32), np.asarray(ref_p, dtype=np.float32)

    def infer_sequence_to_labels_probs(
        self, sequences: List[str], conditional_input=None
    ) -> Optional[np.ndarray]:
        if self._clf_head is None:
            return None  # GUE needs a head — attach it via linear_probe_gue
        _, _, reps = self.infer_sequence_to_sequence(sequences)
        with torch.no_grad():
            x = torch.from_numpy(np.asarray(reps, dtype=np.float32)).to(self.device)
            return F.softmax(self._clf_head(x), dim=-1).cpu().numpy()

    def infer_variant_ref_sequences_to_labels_probs(
        self, variant_sequences, ref_sequences, conditional_input=None
    ):
        return None

    def infer_sequence_to_regression(self, sequences, conditional_input=None):
        return None


ntv3 = SimpleNTv3Model(device=DEVICE, max_length=512)


### Checklist: which methods each task uses

- **BEND expression:** `infer_sequence_to_sequence`, `sequence_pos_to_prob_pos`, `infer_masked_sequence_to_token_probs`
- **GUE promoter (after probe):** `infer_sequence_to_labels_probs` (backed by reps from `infer_sequence_to_sequence`)


### B.1 NTv3 on BEND expression


In [ ]:
# Reuse the same BendVEPExpression task instance / config
ntv3.eval()
bend_scores_ntv3 = bend_task.eval_test_set(ntv3)
print("BEND scores (NTv3 8M):")
for k, v in bend_scores_ntv3.items():
    print(f"  {k}: {v}")


### B.2 Probe + evaluate NTv3 on GUE promoter


In [ ]:
print("Linear probing NTv3 on GUE train split...")
linear_probe_gue(ntv3, gue_task)

ntv3.eval()
gue_scores_ntv3 = gue_task.eval_test_set(ntv3)
print("GUE scores (NTv3 8M + probe):")
for k, v in gue_scores_ntv3.items():
    print(f"  {k}: {v}")


## 4. Takeaways

1. **Copy a skeleton** (`RandomMockModel` or `SimpleNTv3Model`) and replace the forward path with your model.
2. **Return `None`** for methods you do not support — GFMBench-API skips related metrics.
3. **Supervised tasks** need `infer_sequence_to_labels_probs` (or a probe head that provides it). Zero-shot VEP tasks need sequence / masked-token APIs instead.
4. For a full multi-task suite, see `usage_examples/run_benchmark.py`. If model deps conflict with `basic_requirements.txt`, see the README section *Isolated model environments* and `usage_examples/sanity_models/isolated_mock_model.py`.

You are ready to swap in your own weights and evaluate.
